# Dataset preparation

This file is the file for preparing the dataset

## Import dependency and read config

In [3]:
# General imports

import re
import json
from pathlib import Path
from typing import Union, Any

In [4]:
# GitHub API imports

import requests
from datetime import datetime, timedelta
from time import sleep, time

In [5]:
with open('src/config.json', 'r') as file:
    config = json.load(file)

## Setup database path

In [6]:
github_token = config['github_token']
root_database_path = Path(config['root_database'])
package_json_history_path = root_database_path.joinpath('package_json_history')

## Gathering all package.json history based on removed dependency

### Fetch removed dependencies and its dependents (remover) from database

In [7]:
with open(f'{root_database_path}/dependency_removed_with_dependents.json') as file:
    dependency_removed_with_dependents = json.load(file)

In [8]:
dependency_removed_with_dependents

{'safe-buffer': [{'package_name': 'minipass',
   'repo_link': 'https://api.github.com/repos/isaacs/minipass',
   'version_before_change': {'version': '2.9.0',
    'dependencies': {'safe-buffer': '^5.1.2', 'yallist': '^3.0.0'}},
   'version_after_change': {'version': '3.0.0',
    'dependencies': {'yallist': '^4.0.0'}}},
  {'package_name': 'ws',
   'repo_link': 'https://api.github.com/repos/websockets/ws',
   'version_before_change': {'version': '4.1.0',
    'dependencies': {'async-limiter': '~1.0.0', 'safe-buffer': '~5.1.0'}},
   'version_after_change': {'version': '5.0.0',
    'dependencies': {'async-limiter': '~1.0.0'}}},
  {'package_name': 'dns-packet',
   'repo_link': 'https://api.github.com/repos/mafintosh/dns-packet',
   'version_before_change': {'version': '4.2.0',
    'dependencies': {'ip': '^1.1.5', 'safe-buffer': '^5.1.1'}},
   'version_after_change': {'version': '5.0.0',
    'dependencies': {'ip': '^1.1.5'}}},
  {'package_name': 'spdy',
   'repo_link': 'https://api.github.com

In [9]:
dependency_removed_with_dependents_without_version = {}
for package, dependents in dependency_removed_with_dependents.items():
    dependents_name = []
    for dependent in dependents:
        org, repo = dependent['repo_link'].rsplit('/', 2)[1:]
        dependents_name.append(f'{org}/{repo}')

    dependency_removed_with_dependents_without_version[package] = dependents_name

In [10]:
dependency_removed_with_dependents_without_version

{'safe-buffer': ['isaacs/minipass',
  'websockets/ws',
  'mafintosh/dns-packet',
  'indutny/node-spdy',
  'spdy-http2/spdy-transport',
  'nodejs/readable-stream',
  'rvagg/bl',
  'avajs/ava',
  'isaacs/node-tar',
  'mafintosh/sparse-bitfield',
  'npm/make-fetch-happen',
  'npm/ssri',
  'lerna/lerna',
  'npm/npm-registry-fetch',
  'npm/cli'],
 'indent-string': ['sindresorhus/meow'],
 'decamelize': ['sindresorhus/meow', 'yargs/yargs', 'cssnano/cssnano'],
 'map-obj': ['sindresorhus/meow'],
 'object-assign': ['sindresorhus/meow',
  'sindresorhus/yn',
  'sindresorhus/get-stream',
  'sindresorhus/normalize-url',
  'webpack/loader-utils',
  'josdejong/workerpool',
  'facebook/react',
  'eslint/eslint',
  'estools/esrecurse',
  'jaredwray/file-entry-cache',
  'webpack/webpack',
  'webpack/enhanced-resolve',
  'istanbuljs/babel-plugin-istanbul',
  'istanbuljs/test-exclude',
  'sindresorhus/execa',
  'jestjs/jest',
  'facebook/react',
  'facebook/react',
  'sindresorhus/globby',
  'import-js/esl

### Download all version of package.json from GitHub

In [11]:
EMPHZIED = '\033[1m'
ERROR = '\033[31m'
SUCCESS = '\033[32m'
WHITE = '\033[37m'
WARNING = '\033[93m'
INFO = '\033[94m'
PURPLE = '\033[95m'
CYAN = '\033[96m'
ENDC = '\033[0m'

In [12]:
def clean_and_fix_json(broken_json: str) -> Any:
    # Remove any leading unexpected characters before the JSON starts
    broken_json = re.sub(r'^[^\{]*', '', broken_json)

    # Remove any trailing unexpected characters after the JSON ends
    broken_json = re.sub(r'[^\}]*$', '', broken_json)

    # Remove comments
    # Remove single-line comments
    broken_json = re.sub(r'//.*?(\n|$)', '\n', broken_json)
    broken_json = re.sub(r'/\*.*?\*/', '', broken_json,
                         flags=re.DOTALL)  # Remove multi-line comments

    # Remove trailing commas
    broken_json = re.sub(r',\s*([\]}])', r'\1', broken_json)

    # Attempt to add missing commas
    broken_json = re.sub(r'([}\]"\'\w])\s*([\[{])', r'\1,\2', broken_json)

    # Ensure keys are quoted
    broken_json = re.sub(r'([{,]\s*)(\w+)(\s*:)', r'\1"\2"\3', broken_json)

    # Attempt to parse the JSON
    try:
        return json.loads(broken_json)
    except json.JSONDecodeError as e:
        print(f'Failed to parse JSON: {e}')
        return None

In [27]:
def log_error_limit_reached(package_name: str, api: str, status_code: int, wait_time: timedelta) -> None:
    print(
        f'{ERROR}ERROR{ENDC}, with {WARNING}{status_code}{ENDC} when getting {WARNING}{package_name}{ENDC} from {WARNING}{api}{ENDC}')
    while wait_time > 0:
        if wait_time != 1:
            print(f"out of x-ratelimit: wait {WARNING}%02d:%02d:%02d{ENDC} to get the data again\r" %
                (wait_time // 3600, wait_time // 60, wait_time % 60), end="")
            wait_time -= 1
            sleep(1)

        else:
            print(f"out of x-ratelimit: wait {WARNING}%02d:%02d:%02d{ENDC} to get the data again\n" %
        (wait_time // 3600, wait_time // 60, wait_time % 60), end="")
            sleep(3)

def request_api(
        api: str,
        package_name: str, # * For logging purpose
        headers: Union[dict, str, None] = None,
        spare_api: Union[str, None] = None
) -> Union[dict, bool]:
    if type(headers) is str:
        headers = {"Authorization": f"Bearer {headers}"}
        
    session = requests.Session()
    session.headers.update(headers)
    res = session.get(api)
    # res = requests.get(api, headers=headers)

    if 'x-ratelimit-remaining' in res.headers.keys():
        requests_left = res.headers['x-ratelimit-remaining']
    else:
        requests_left = None

    if 'x-ratelimit-reset' in res.headers.keys():
        time_stamp = datetime.fromtimestamp(time())
        reset_time = datetime.fromtimestamp(int(res.headers['x-ratelimit-reset']))
        duration = reset_time - time_stamp

    else:
        duration = 60

        match res.status_code:
            case 401:
                print(f'{ERROR}ERROR{ENDC}, Unauthorized. Please check your GitHub token.')

            case 404:
                print(f'{ERROR}ERROR{ENDC}, not found when getting {WARNING}{package_name}{ENDC} from {WARNING}{api}{ENDC}')
                if spare_api is not None:
                    print(f'{WARNING}Try{ENDC} with {WARNING}{spare_api}{ENDC}')
                else:
                    print(f'{ERROR}No spare api{ENDC} for {WARNING}{package_name}{ENDC}')
                    return None, requests_left
                
                res = session.get(spare_api)

                match res.status_code:
                    case 403 | 429:
                        max_retries = 3
                        for try_count in range(max_retries):
                            wait_time = duration.total_seconds()
                            log_error_limit_reached(package_name, api, res.status_code, wait_time)

                            try:
                                res = session.get(api)
                                res = res.json()
                            except requests.exceptions.RequestException as e:
                                print(f'{ERROR}ERROR{ENDC}, with {WARNING}{e}{ENDC}')
                                return None, requests_left
                            return res, requests_left
                    case _:
                        if res.status_code != 200:
                            print(f'{ERROR}ERROR{ENDC}, with {WARNING}{res.status_code}{ENDC} when getting {WARNING}{package_name}{ENDC} from {WARNING}{api}{ENDC}')
                            return None, requests_left
                        
                        res = res.json()
                        print(f'{SUCCESS}Success{ENDC}, with {WARNING}{res.status_code}{ENDC} when getting {WARNING}{package_name}{ENDC} from {WARNING}{api}{ENDC}')
                        return res, requests_left

            case 403 | 429:
                wait_time = duration.total_seconds()
                log_error_limit_reached(package_name, api, res.status_code, wait_time)

                try:
                    res = session.get(api)
                    res = res.json()
                except requests.exceptions.RequestException as e:
                    print(f'{ERROR}ERROR{ENDC}, with {WARNING}{e}{ENDC}')
                    return e.args
                return res

            case 422:
                print(f'{ERROR}ERROR{ENDC}, exceed limit requests with {WARNING}{res.status_code}{ENDC} when getting {WARNING}{package_name}{ENDC} from {WARNING}{api}{ENDC}')
                return None, requests_left

            case _:
                if res.status_code != 200:
                    print(f'{ERROR}ERROR{ENDC}, with {WARNING}{res.status_code}{ENDC} when getting {WARNING}{package_name}{ENDC} from {WARNING}{api}{ENDC}')
                    return None, requests_left
                
                try:
                    res = res.json()
                except json.decoder.JSONDecodeError as e:
                    print(f'{WARNING}Debugging step{ENDC}')
                    print(api)
                    print(res.text)
                    result = clean_and_fix_json(res.text)
                    if result is None:
                        print(res.text)
                        return None, requests_left

                return res, requests_left
        

In [26]:
def requests_github_api(
    headers: dict,
    url: str,
    # **FOR LOGGING**: logging the name of the file and repository
    org: str,
    repo_name: str,
) -> Union[dict, None]:
    """
    Makes a GET request to the GitHub API and handles rate limiting.

    Args:
        headers (dict): The headers to include in the request.
        url (str): The URL to send the request to.
        org (str): The organization name for logging purposes.
        repo_name (str): The repository name for logging purposes.

    Returns:
        res (Union[dict, None]): The JSON response from the API if successful, None otherwise.
        requests_left (int): The number of requests left in the rate limit.
    """
    session = requests.Session()
    session.headers.update(headers)

    response = session.get(url)
    current_time = int(time())

    if 'X-RateLimit-Remaining' in response.headers:
        requests_left = int(response.headers['X-RateLimit-Remaining'])
    else:
        requests_left = None

    if response.status_code == 403 or response.status_code == 429:
        res = response.json()
        if res['message'] == 'Repository access blocked':
            print(f'{ERROR}Access blocked for {org}:{repo_name}{ENDC}, skip to the next one')
            return None, requests_left

        print(f'{ERROR}Rate limit exceeded{ENDC}')
        reset_time = int(response.headers['X-RateLimit-Reset'])
        remaining_time = reset_time - current_time

        while remaining_time > 0:
            print(f'\rWaiting for {remaining_time} seconds', end='')
            sleep(1)
            remaining_time -= 1

        try:
            response = session.get(url, headers=headers)
        except requests.exceptions.RequestException as e:
            print(json.dumps(dict(response.headers), indent=4))
            print(f'{ERROR}Error at {org}:{repo_name}{ENDC}')
            print(f'{ERROR}Error: {e}{ENDC}')
            return None, requests_left

    elif response.status_code != 200:
        print(f'{ERROR}Error: {response.status_code} when getting requirements.txt from {org}:{repo_name}{ENDC}')
        print(f'{ERROR}The URL that caused the error: {url}{ENDC}')
        return None, requests_left

    try:
        res = response.json()
    except:
        res = response.text

    return res, requests_left


In [28]:
def get_package_json_history(
        org: str,
        repo: str,
        github_token: str,
        save_path: Path,
        update: bool = False,
        spare_api: Union[str, None] = None,
) -> None:
    saved_files = save_path.glob('*.json')
    saved_files = [str(file) for file in saved_files]
    # print(list(saved_files))
    # print(json.dumps(list(saved_files), indent=4))

    headers = {"Authorization": f"Bearer {github_token}"}

    page = 1
    no_download = True
    while True:
        api = f'https://api.github.com/repos/{org}/{repo}/commits?path=package.json&per_page=100&page={page}'

        res = request_api(api, f'{org}:{repo}', headers, spare_api)
        if res is None or len(res) == 0:
            break

        for commit in res:
            try:
                commit_date = commit['commit']['author']['date']
                commit_date = datetime.strptime(commit_date, '%Y-%m-%dT%H:%M:%SZ')
                commit_date = commit_date.strftime('%Y-%m-%d')
            except:
                try:
                    print(commit['commit']['author'])
                except:
                    try:
                        print(commit['commit'])
                    except:
                        print(commit)

            commit_sha = commit['sha']

            if not save_path.exists():
                save_path.mkdir(parents=True)
            save_file_name = f'{save_path}/{commit_date}_{commit_sha}.json'

            # print(save_file_name)
            if save_file_name in saved_files and not update:
                print(f'{WARNING}Found{ENDC} {WARNING}{save_file_name}{ENDC} the folder {WARNING}{org}:{repo}{ENDC}, skip to the next commit')
                continue

            description_api = f'https://api.github.com/repos/{org}/{repo}/contents/package.json?ref={commit_sha}'

            description_res = request_api(description_api, f'{org}:{repo}', headers, spare_api)
            if description_res is None:
                break

            with open(save_file_name, 'w') as file:
                file.write(json.dumps(description_res, indent=4))

            no_download = False

        page += 1
    
    if no_download:
        print(f'{WARNING}Already download{ENDC} all version of package.json of {WARNING}{org}:{repo}{ENDC}, skip to the next dependent')

In [30]:
for removed_depednency, dependents in dependency_removed_with_dependents_without_version.items():
    print(f'{EMPHZIED}Dependency{ENDC}: {WARNING}{removed_depednency}{ENDC}')

    for dependent in dependents:
        print(f'    {EMPHZIED}Dependent{ENDC}: {INFO}{dependent}{ENDC}')
        org, repo = dependent.split('/')
        # for folder in saved_folders:
        #     if org in folder.name and repo in folder.name:
        #         print(f'{SUCCESS}Found{ENDC} the folder {WARNING}{folder.name}{ENDC}, skip to the next dependent')
        #         continue

        print(f'{INFO}Getting{ENDC} history of package.json of {WARNING}{dependent}{ENDC}')
        save_path = package_json_history_path.joinpath(f'{org}:{repo}')
        get_package_json_history(
            org=org,
            repo=repo,
            github_token=github_token,
            save_path=save_path,
            update=False
        )


Dependency: safe-buffer
    Dependent: isaacs/minipass
Getting history of package.json of isaacs/minipass
Already download all version of package.json of isaacs:minipass, skip to the next dependent
    Dependent: websockets/ws
Getting history of package.json of websockets/ws
Already download all version of package.json of websockets:ws, skip to the next dependent
    Dependent: mafintosh/dns-packet
Getting history of package.json of mafintosh/dns-packet
Already download all version of package.json of mafintosh:dns-packet, skip to the next dependent
    Dependent: indutny/node-spdy
Getting history of package.json of indutny/node-spdy
Already download all version of package.json of indutny:node-spdy, skip to the next dependent
    Dependent: spdy-http2/spdy-transport
Getting history of package.json of spdy-http2/spdy-transport
Already download all version of package.json of spdy-http2:spdy-transport, skip to the next dependent
    Dependent: nodejs/readable-stream
Getting history of pack